<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/14PREZENTACJA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_prezentacja.py

from __future__ import annotations

import csv
import json

from dataclasses import dataclass, field
from enum import StrEnum, verify, UNIQUE
from pathlib import Path
from typing import (
    Any,
    Callable,
    Final,
    Protocol,
)


FOLDER_DANYCH: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)

PLIK_CSV: Final[Path] = (
    FOLDER_DANYCH
    / "wyniki_koncowe.csv"
)


@verify(UNIQUE)
class StatusPrezentacji(StrEnum):
    PASSED = "passed"
    FAILED = "failed"


@verify(UNIQUE)
class TypKomunikatu(StrEnum):
    INFO = "info"
    SUCCESS = "success"
    WARNING = "warning"
    ERROR = "error"


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True,
)
class WynikDoPrezentacji:

    ticker: str

    status: StatusPrezentacji

    wynik_punktowy: float

    pozycja_w_rankingu: int | None

    poziom_jakosci: str | None

    liczba_filtrow: int

    liczba_filtrow_passed: int

    liczba_filtrow_failed: int

    filtry: tuple[
        str,
        ...
    ] = field(
        default_factory=tuple,
        compare=False,
    )


class ObserverProtocol(Protocol):

    def notify(
        self,
        komunikat: str,
        typ: TypKomunikatu,
    ) -> None:
        ...


ObserverCallable = Callable[
    [
        str,
        TypKomunikatu,
    ],
    None,
]


class ConsoleObserver:

    def notify(
        self,
        komunikat: str,
        typ: TypKomunikatu,
    ) -> None:

        print(
            f"[{typ.value.upper()}]",
            komunikat,
        )


class PrezentacjaTerminalowa:

    def pokaz_wynik(
        self,
        wynik: WynikDoPrezentacji,
    ) -> None:

        print(
            "\n"
            + "=" * 60
        )

        print(
            "WYNIK KONCOWY"
        )

        print(
            "=" * 60
        )

        print(
            "Ticker:",
            wynik.ticker,
        )

        print(
            "Status:",
            wynik.status.value.upper(),
        )

        print(
            "Punktacja:",
            round(
                wynik.wynik_punktowy,
                2,
            ),
        )

        print(
            "Pozycja w rankingu:",
            (
                wynik.pozycja_w_rankingu
                if wynik.pozycja_w_rankingu
                is not None
                else "brak"
            ),
        )

        print(
            "Poziom jakosci:",
            (
                wynik.poziom_jakosci
                if wynik.poziom_jakosci
                is not None
                else "brak"
            ),
        )

        print(
            "Liczba filtrow:",
            wynik.liczba_filtrow,
        )

        print(
            "PASSED:",
            wynik.liczba_filtrow_passed,
        )

        print(
            "FAILED:",
            wynik.liczba_filtrow_failed,
        )

        print(
            "\nFILTRY:"
        )

        if not wynik.filtry:

            print(
                "brak wynikow filtrow"
            )

        else:

            for filtr in wynik.filtry:

                print(
                    "-",
                    filtr,
                )

        print(
            "=" * 60
        )

    def pokaz_ranking(
        self,
        wyniki: list[
            WynikDoPrezentacji
        ],
    ) -> None:

        print(
            "\n"
            + "=" * 60
        )

        print(
            "PODSUMOWANIE WSZYSTKICH SPOLEK"
        )

        print(
            "=" * 60
        )

        posortowane: list[
            WynikDoPrezentacji
        ] = sorted(
            wyniki,
            key=lambda wynik: (
                wynik.pozycja_w_rankingu
                if wynik.pozycja_w_rankingu
                is not None
                else 999999
            ),
        )

        for wynik in posortowane:

            pozycja: str = (
                str(
                    wynik.pozycja_w_rankingu
                )
                if wynik.pozycja_w_rankingu
                is not None
                else "-"
            )

            print(
                f"{pozycja:>3} "
                f"{wynik.ticker:<10} "
                f"{wynik.status.value.upper():<8} "
                f"punkty="
                f"{wynik.wynik_punktowy:.2f} "
                f"jakosc="
                f"{wynik.poziom_jakosci}"
            )

        print(
            "=" * 60
        )


class CsvExporter:

    def __init__(
        self,
        plik: Path,
    ) -> None:

        self.plik = plik

        self._file = None

    def __enter__(
        self,
    ) -> CsvExporter:

        self.plik.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        self._file = open(
            self.plik,
            "w",
            encoding="utf-8",
            newline="",
        )

        return self

    def __exit__(
        self,
        exc_type,
        exc_value,
        traceback,
    ) -> bool:

        if self._file is not None:

            self._file.close()

        return False

    def eksportuj(
        self,
        wyniki: list[
            WynikDoPrezentacji
        ],
    ) -> None:

        if self._file is None:

            raise RuntimeError(
                "CsvExporter musi byc "
                "uzyty jako context manager"
            )

        kolumny: tuple[
            str,
            ...
        ] = (
            "pozycja",
            "ticker",
            "status",
            "wynik_punktowy",
            "poziom_jakosci",
            "liczba_filtrow",
            "liczba_passed",
            "liczba_failed",
            "filtry",
        )

        writer = csv.DictWriter(
            self._file,
            fieldnames=kolumny,
        )

        writer.writeheader()

        posortowane: list[
            WynikDoPrezentacji
        ] = sorted(
            wyniki,
            key=lambda wynik: (
                wynik.pozycja_w_rankingu
                if wynik.pozycja_w_rankingu
                is not None
                else 999999
            ),
        )

        for wynik in posortowane:

            writer.writerow(
                {
                    "pozycja":
                        (
                            wynik.pozycja_w_rankingu
                            if wynik.pozycja_w_rankingu
                            is not None
                            else ""
                        ),

                    "ticker":
                        wynik.ticker,

                    "status":
                        wynik.status.value,

                    "wynik_punktowy":
                        wynik.wynik_punktowy,

                    "poziom_jakosci":
                        (
                            wynik.poziom_jakosci
                            if wynik.poziom_jakosci
                            is not None
                            else ""
                        ),

                    "liczba_filtrow":
                        wynik.liczba_filtrow,

                    "liczba_passed":
                        wynik.liczba_filtrow_passed,

                    "liczba_failed":
                        wynik.liczba_filtrow_failed,

                    "filtry":
                        " | ".join(
                            wynik.filtry
                        ),
                }
            )


class WynikRepository:

    def __init__(
        self,
        folder: Path,
    ) -> None:

        self.folder = folder

    def pobierz(
        self,
        ticker: str,
    ) -> WynikDoPrezentacji:

        ticker = (
            ticker
            .strip()
            .upper()
        )

        plik: Path = (
            self.folder
            / (
                f"{ticker}"
                "_wynik_koncowy.json"
            )
        )

        if not plik.exists():

            raise FileNotFoundError(
                f"brak wyniku koncowego: "
                f"{plik}"
            )

        with open(
            plik,
            "r",
            encoding="utf-8",
        ) as f:

            dane: Any = json.load(
                f
            )

        if not isinstance(
            dane,
            dict,
        ):

            raise ValueError(
                "wynik koncowy "
                "musi byc obiektem JSON"
            )

        return self._mapuj(
            dane
        )

    def _mapuj(
        self,
        dane: dict[str, Any],
    ) -> WynikDoPrezentacji:

        try:

            status = StatusPrezentacji(
                str(
                    dane["status"]
                )
            )

        except ValueError as e:

            raise ValueError(
                f"nieznany status: "
                f"{dane['status']}"
            ) from e

        filtry_raw: Any = (
            dane.get(
                "wyniki_filtrow",
                [],
            )
        )

        if not isinstance(
            filtry_raw,
            list,
        ):

            raise ValueError(
                "wyniki_filtrow "
                "musza byc lista"
            )

        filtry: list[str] = []

        passed: int = 0
        failed: int = 0

        for rekord in filtry_raw:

            if not isinstance(
                rekord,
                dict,
            ):

                raise ValueError(
                    "wynik filtra "
                    "musi byc dict"
                )

            typ: str = str(
                rekord.get(
                    "typ",
                    "nieznany",
                )
            )

            status_filtra: str = str(
                rekord.get(
                    "status",
                    "",
                )
            )

            wartosc: float = float(
                rekord.get(
                    "wartosc",
                    0.0,
                )
            )

            prog: float = float(
                rekord.get(
                    "prog",
                    0.0,
                )
            )

            if (
                status_filtra
                == StatusPrezentacji.PASSED.value
            ):

                passed += 1

            elif (
                status_filtra
                == StatusPrezentacji.FAILED.value
            ):

                failed += 1

            filtry.append(
                (
                    f"{typ}: "
                    f"{status_filtra} "
                    f"(wartosc={wartosc:.4f}, "
                    f"prog={prog})"
                )
            )

        poziom_raw: Any = (
            dane.get(
                "poziom_jakosci",
                {},
            )
        )

        poziom: str | None = None

        if isinstance(
            poziom_raw,
            dict,
        ):

            nazwa: Any = (
                poziom_raw.get(
                    "nazwa"
                )
            )

            if nazwa is not None:

                poziom = str(
                    nazwa
                )

        pozycja_raw: Any = (
            dane.get(
                "pozycja_w_rankingu"
            )
        )

        pozycja: int | None = None

        if pozycja_raw is not None:

            pozycja = int(
                pozycja_raw
            )

        return WynikDoPrezentacji(
            ticker=str(
                dane["ticker"]
            ).strip().upper(),

            status=status,

            wynik_punktowy=float(
                dane.get(
                    "wynik_punktowy",
                    0.0,
                )
            ),

            pozycja_w_rankingu=(
                pozycja
            ),

            poziom_jakosci=(
                poziom
            ),

            liczba_filtrow=len(
                filtry
            ),

            liczba_filtrow_passed=(
                passed
            ),

            liczba_filtrow_failed=(
                failed
            ),

            filtry=tuple(
                filtry
            ),
        )


class PrezentacjaService:

    def __init__(
        self,
        *,
        repository: WynikRepository,
        presenter: PrezentacjaTerminalowa,
        observers: list[
            ObserverCallable
        ] | None = None,
    ) -> None:

        self.repository = repository

        self.presenter = presenter

        self.observers = (
            list(observers)
            if observers is not None
            else []
        )

    def notify(
        self,
        komunikat: str,
        typ: TypKomunikatu = (
            TypKomunikatu.INFO
        ),
    ) -> None:

        for observer in self.observers:

            observer(
                komunikat,
                typ,
            )

    def pobierz_wyniki(
        self,
        tickery: list[str],
    ) -> list[
        WynikDoPrezentacji
    ]:

        wyniki: list[
            WynikDoPrezentacji
        ] = []

        for ticker in tickery:

            self.notify(
                (
                    "wczytywanie wyniku "
                    f"{ticker}"
                ),
                TypKomunikatu.INFO,
            )

            try:

                wynik = (
                    self.repository
                    .pobierz(
                        ticker
                    )
                )

                wyniki.append(
                    wynik
                )

                self.notify(
                    (
                        f"{ticker} "
                        "wczytany poprawnie"
                    ),
                    TypKomunikatu.SUCCESS,
                )

            except Exception as e:

                self.notify(
                    (
                        f"{ticker}: "
                        f"{e}"
                    ),
                    TypKomunikatu.ERROR,
                )

        return wyniki

    def pokaz(
        self,
        wyniki: list[
            WynikDoPrezentacji
        ],
    ) -> None:

        for wynik in wyniki:

            self.presenter.pokaz_wynik(
                wynik
            )

        self.presenter.pokaz_ranking(
            wyniki
        )

    def eksportuj_csv(
        self,
        wyniki: list[
            WynikDoPrezentacji
        ],
        plik: Path = PLIK_CSV,
    ) -> Path:

        self.notify(
            "eksport do CSV",
            TypKomunikatu.INFO,
        )

        with CsvExporter(
            plik
        ) as exporter:

            exporter.eksportuj(
                wyniki
            )

        self.notify(
            (
                "zapisano CSV: "
                f"{plik}"
            ),
            TypKomunikatu.SUCCESS,
        )

        return plik


class PresentationFacade:

    def __init__(
        self,
        service: PrezentacjaService,
    ) -> None:

        self.service = service

    def wykonaj(
        self,
        tickery: list[str],
    ) -> Path:

        if not tickery:

            raise ValueError(
                "lista tickerow "
                "nie moze byc pusta"
            )

        wyniki: list[
            WynikDoPrezentacji
        ] = (
            self.service
            .pobierz_wyniki(
                tickery
            )
        )

        if not wyniki:

            raise ValueError(
                "nie znaleziono "
                "wynikow do prezentacji"
            )

        self.service.pokaz(
            wyniki
        )

        return (
            self.service
            .eksportuj_csv(
                wyniki
            )
        )


def run() -> None:

    tekst: str = input(
        "podaj tickery "
        "oddzielone przecinkami: "
    )

    tickery: list[str] = [

        ticker.strip().upper()

        for ticker
        in tekst.split(",")

        if ticker.strip()
    ]

    if not tickery:

        raise ValueError(
            "nie podano tickerow"
        )

    repository = WynikRepository(
        folder=FOLDER_DANYCH
    )

    presenter = (
        PrezentacjaTerminalowa()
    )

    observer = ConsoleObserver()

    service = PrezentacjaService(
        repository=repository,
        presenter=presenter,
        observers=[
            observer.notify
        ],
    )

    facade = PresentationFacade(
        service=service
    )

    plik: Path = facade.wykonaj(
        tickery
    )

    print(
        "\nEKSPORT ZAKONCZONY"
    )

    print(
        plik
    )

    print(
        "\nMODUL PREZENTACJI "
        "DZIALA POPRAWNIE"
    )